## Support Displacement of a Frame


In [77]:
# -------------------------
# Import Libraries and Helper Functions
# -------------------------
# This slide imports the required Python libraries and adds the
# course helper-code directory to the Python path.

import sys
import os
import numpy as np
import pandas as pd

# Get current notebook directory
current_dir = os.getcwd()

# Go up two levels, then into Code/L8
helpers_path = os.path.abspath(
    os.path.join(current_dir, "..", "..", "Code", "L8")
)

sys.path.append(helpers_path)

In [78]:
import numpy as np

from helper_funcs_dsm import (
    assemble_global_stiffness_and_fef,
    partition_system,
    assemble_global_displacements,
    assemble_global_forces
)
from helper_funcs_output import (
    print_element,
    print_dsm_results,
    print_matrix_scaled
)


In [79]:
# -------------------------
# Problem data (hard-coded)
# -------------------------
E = 200e6          # kPa
I = 762e-6         # m^4

In [80]:
import numpy as np

def frame_stiffness_2d(E, A, I, L):
    """
    Return the 6x6 local stiffness matrix for a 2D frame element.
    """

    k = np.array([
        [ A*E/L,          0,              0,        -A*E/L,          0,              0],
        [     0,   12*E*I/L**3,    6*E*I/L**2,           0, -12*E*I/L**3,    6*E*I/L**2],
        [     0,    6*E*I/L**2,     4*E*I/L,              0,  -6*E*I/L**2,     2*E*I/L],
        [-A*E/L,          0,              0,         A*E/L,          0,              0],
        [     0, -12*E*I/L**3,   -6*E*I/L**2,           0,  12*E*I/L**3,   -6*E*I/L**2],
        [     0,    6*E*I/L**2,     2*E*I/L,              0,  -6*E*I/L**2,     4*E*I/L]
    ], dtype=float)

    return k

def frame_transformation_2d(theta_deg):
    """
    Return the 6x6 transformation matrix for a 2D frame element.

    DOF order:
    [u1, v1, θ1, u2, v2, θ2]

    Parameters
    ----------
    theta_deg : float
        Element angle measured from global +X axis (degrees)

    Returns
    -------
    T : (6,6) ndarray
        Transformation matrix (global → local)
    """

    theta = np.radians(theta_deg)

    c = np.cos(theta)
    s = np.sin(theta)

    T = np.array([
        [ c,  s, 0,  0,  0, 0],
        [-s,  c, 0,  0,  0, 0],
        [ 0,  0, 1,  0,  0, 0],
        [ 0,  0, 0,  c,  s, 0],
        [ 0,  0, 0, -s,  c, 0],
        [ 0,  0, 0,  0,  0, 1]
    ], dtype=float)

    return T


In [81]:
# -------------------------
# Member 1
# -------------------------
L1 = 10.0            # m
A1 = 0.013
deg1 = np.degrees(np.arctan2(8,-6))

k1 = frame_stiffness_2d(E, A1, I, L1)

# Transformation matrix
T1 = frame_transformation_2d(deg1)

# Mapping
m1 = np.array([4,5,6,1,2,3])

print_matrix_scaled(k1, scale=1000, decimals=2, col_width=8)

print("T1:", T1)

print_matrix_scaled(T1.T @ k1 @ T1, scale=1000, decimals=2, col_width=8)

K = 1e+03 ×
01 |   260.00     0.00     0.00  -260.00     0.00     0.00
02 |     0.00     1.83     9.14     0.00    -1.83     9.14
03 |     0.00     9.14    60.96     0.00    -9.14    30.48
04 |  -260.00     0.00     0.00   260.00     0.00     0.00
05 |     0.00    -1.83    -9.14     0.00     1.83    -9.14
06 |     0.00     9.14    30.48     0.00    -9.14    60.96
T1: [[-0.6  0.8  0.   0.   0.   0. ]
 [-0.8 -0.6  0.   0.   0.   0. ]
 [ 0.   0.   1.   0.   0.   0. ]
 [ 0.   0.   0.  -0.6  0.8  0. ]
 [ 0.   0.   0.  -0.8 -0.6  0. ]
 [ 0.   0.   0.   0.   0.   1. ]]
K = 1e+03 ×
01 |    94.77  -123.92    -7.32   -94.77   123.92    -7.32
02 |  -123.92   167.06    -5.49   123.92  -167.06    -5.49
03 |    -7.32    -5.49    60.96     7.32     5.49    30.48
04 |   -94.77   123.92     7.32    94.77  -123.92     7.32
05 |   123.92  -167.06     5.49  -123.92   167.06     5.49
06 |    -7.32    -5.49    30.48     7.32     5.49    60.96


In [82]:
# Local fixed-end vector for member 1
P1 = -125

v = P1/2
m = P1*L1/8

Qf1 = np.array([0,v,m,0,v,-m])
print(Qf1)
print(T1.T @ Qf1)

[   0.    -62.5  -156.25    0.    -62.5   156.25]
[  50.     37.5  -156.25   50.     37.5   156.25]


In [83]:
# -------------------------
# Member 2
# -------------------------
L2 = 12.0
A2 = 0.013
deg2 = 0

k2 = frame_stiffness_2d(E, A2, I, L2)

# Transformation matrix
T2 = frame_transformation_2d(deg2)

# Mapping
m2 = np.array([4,5,6,7,8,9])


print_matrix_scaled(k2, scale=1000, decimals=2, col_width=8)

print("T2:", T2)

print_matrix_scaled(T2.T @ k2 @ T2, scale=1000, decimals=2, col_width=8)


K = 1e+03 ×
01 |   216.67     0.00     0.00  -216.67     0.00     0.00
02 |     0.00     1.06     6.35     0.00    -1.06     6.35
03 |     0.00     6.35    50.80     0.00    -6.35    25.40
04 |  -216.67     0.00     0.00   216.67     0.00     0.00
05 |     0.00    -1.06    -6.35     0.00     1.06    -6.35
06 |     0.00     6.35    25.40     0.00    -6.35    50.80
T2: [[ 1.  0.  0.  0.  0.  0.]
 [-0.  1.  0.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.]
 [ 0.  0.  0. -0.  1.  0.]
 [ 0.  0.  0.  0.  0.  1.]]
K = 1e+03 ×
01 |   216.67     0.00     0.00  -216.67     0.00     0.00
02 |     0.00     1.06     6.35     0.00    -1.06     6.35
03 |     0.00     6.35    50.80     0.00    -6.35    25.40
04 |  -216.67     0.00     0.00   216.67     0.00     0.00
05 |     0.00    -1.06    -6.35     0.00     1.06    -6.35
06 |     0.00     6.35    25.40     0.00    -6.35    50.80


In [84]:
# Local fixed-end vector for member 1
P2 = 24

v = P2*L2/2
m = P2*L2**2/12

Qf2 = np.array([0,v,m,0,v,-m])
print(Qf2)

print(T2.T @ Qf2)

[   0.  144.  288.    0.  144. -288.]
[   0.  144.  288.    0.  144. -288.]


In [85]:
# -------------------------
# Initialize Global System
# -------------------------
# This slide sets up the global variables needed for assembly of the
# Direct Stiffness Method system.

k_list = [k1, k2]
T_list = [T1, T2]
Qf_list = [Qf1, Qf2]
map_list = [m1, m2]   # 1-based DOF maps

# Global system size (still using 1-based maps here)
ndof = int(np.max(np.concatenate(map_list)))

# Initialize Applied Load
F_global = np.zeros(ndof, dtype=float)

# Initialize Global Displacement
u_global = np.zeros(ndof, dtype=float)

In [86]:
# -------------------------
# User-Specified Boundary Conditions
# -------------------------
# This slide defines the external loads and support conditions.

# Applied Load (0-indexed)
F_global[6-1] = -150 #kN*m

# Restrained DOFs
dof_restrained_1based = np.array([1, 2, 3, 5, 7, 8], dtype=int)

# # Fictitious restrained DOFs
# dof_fictitious_1based = np.array([], dtype=int)

# # Combine and sort
# dof_restrained_1based = np.sort(
#     np.concatenate((dof_restrained_1based, dof_fictitious_1based))
# )

print(F_global)

[   0.    0.    0.    0.    0. -150.    0.    0.    0.]


In [87]:
# -------------------------
# Assemble Global Stiffness System
# -------------------------
# This step assembles the global stiffness matrix and the global
# fixed-end force vector from all elements.

K_global, F_fef_global = assemble_global_stiffness_and_fef(
    ndof,
    k_list,
    T_list,
    Qf_list,
    map_list
    )

print(F_fef_global)

[  50.     37.5   156.25   50.    181.5   131.75    0.    144.   -288.  ]


In [88]:
# -------------------------
# Display Global Stiffness Matrix
# -------------------------
# Print the assembled global stiffness matrix in a readable format.
# The matrix is scaled for clearer presentation in the slides.

print_matrix_scaled(K_global, scale=1000, decimals=2, col_width=8)

K = 1e+03 ×
01 |    94.77  -123.92     7.32   -94.77   123.92     7.32     0.00     0.00     0.00
02 |  -123.92   167.06     5.49   123.92  -167.06     5.49     0.00     0.00     0.00
03 |     7.32     5.49    60.96    -7.32    -5.49    30.48     0.00     0.00     0.00
04 |   -94.77   123.92    -7.32   311.44  -123.92    -7.32  -216.67     0.00     0.00
05 |   123.92  -167.06    -5.49  -123.92   168.12     0.86     0.00    -1.06     6.35
06 |     7.32     5.49    30.48    -7.32     0.86   111.76     0.00    -6.35    25.40
07 |     0.00     0.00     0.00  -216.67     0.00     0.00   216.67     0.00     0.00
08 |     0.00     0.00     0.00     0.00    -1.06    -6.35     0.00     1.06    -6.35
09 |     0.00     0.00     0.00     0.00     6.35    25.40     0.00    -6.35    50.80


In [89]:
# -------------------------
# Partition Global System
# -------------------------
# The global stiffness equation is partitioned into free and restrained
# degrees of freedom based on the support conditions.

(
    K_ff,
    K_fr,
    K_rf,
    K_rr,
    f_f,
    f_r,
    u_r,
    f_fef_f,
    f_fef_r,
    free_dofs,
    restrained_dofs,
) = partition_system(
    K_global,
    F_global,
    u_global,
    F_fef_global,
    dof_restrained_1based
)

print_matrix_scaled(K_ff, scale=1000, decimals=2, col_width=8)
print_matrix_scaled(K_rr, scale=1000, decimals=2, col_width=8)

print_matrix_scaled(K_rf, scale=1000, decimals=2, col_width=8)
print_matrix_scaled(K_fr, scale=1000, decimals=2, col_width=8)

print(f_f, f_r)
print(f_fef_f, f_fef_r)




K = 1e+03 ×
01 |   311.44    -7.32     0.00
02 |    -7.32   111.76    25.40
03 |     0.00    25.40    50.80
K = 1e+03 ×
01 |    94.77  -123.92     7.32   123.92     0.00     0.00
02 |  -123.92   167.06     5.49  -167.06     0.00     0.00
03 |     7.32     5.49    60.96    -5.49     0.00     0.00
04 |   123.92  -167.06    -5.49   168.12     0.00    -1.06
05 |     0.00     0.00     0.00     0.00   216.67     0.00
06 |     0.00     0.00     0.00    -1.06     0.00     1.06
K = 1e+03 ×
01 |   -94.77     7.32     0.00
02 |   123.92     5.49     0.00
03 |    -7.32    30.48     0.00
04 |  -123.92     0.86     6.35
05 |  -216.67     0.00     0.00
06 |     0.00    -6.35    -6.35
K = 1e+03 ×
01 |   -94.77   123.92    -7.32  -123.92  -216.67     0.00
02 |     7.32     5.49    30.48     0.86     0.00    -6.35
03 |     0.00     0.00     0.00     6.35     0.00    -6.35
[   0. -150.    0.] [0. 0. 0. 0. 0. 0.]
[  50.    131.75 -288.  ] [ 50.    37.5  156.25 181.5    0.   144.  ]


In [90]:
np.linalg.inv(K_ff)

array([[ 3.21650040e-06,  2.37526183e-07, -1.18763092e-07],
       [ 2.37526183e-07,  1.01124324e-05, -5.05621619e-06],
       [-1.18763092e-07, -5.05621619e-06,  2.22131475e-05]])

In [91]:
# -------------------------
# Solve Partitioned System
# -------------------------
# Solve for the unknown displacements at the free DOFs,
# compute the reactions at restrained DOFs, and then
# reconstruct the complete global displacement and force vectors.

rhs = f_f - f_fef_f - K_fr @ u_r
u_f = np.linalg.solve(K_ff, rhs)

print(u_f)

[-0.00026195 -0.00431724  0.00782791]


In [92]:
u_global = assemble_global_displacements(
    u_f,
    u_r,
    free_dofs,
    restrained_dofs
    )

In [93]:
# -------------------------
# Solve Partitioned System
# -------------------------
# Solve for the unknown displacements at the free DOFs,
# compute the reactions at restrained DOFs, and then
# reconstruct the complete global displacement and force vectors.

F_r = K_rf @ u_f + K_rr @ u_r + f_fef_r

f_global_complete = assemble_global_forces(
    f_f,
    F_r,
    free_dofs,
    restrained_dofs
    )

print(np.round(f_global_complete, 2))

# -------------------------
# Print Support Reactions
# -------------------------
print(np.round(f_global_complete[dof_restrained_1based - 1], 2))

[  43.24  -18.65   26.58    0.    259.94 -150.     56.76  121.71    0.  ]
[ 43.24 -18.65  26.58 259.94  56.76 121.71]


In [94]:
# -------------------------
# Display DSM Results
# -------------------------
# Print the final global displacement vector and force vector.

print_dsm_results(
    u_global,
    f_global_complete,
    dof_restrained_1based,
    member_type="frame",
    disp_in_mm=True,
    dec=4,
    rad_dec=6,
)

 DOF  Type Status Disp (mm / rad) Load (kN / kN·m)
   1   u_x  Fixed          0.0000          43.2438
   2   u_y  Fixed          0.0000         -18.6478
   3 theta  Fixed        0.000000          26.5766
   4   u_x   Free         -0.2620           0.0000
   5   u_y  Fixed          0.0000         259.9405
   6 theta   Free       -0.004317        -150.0000
   7   u_x  Fixed          0.0000          56.7562
   8   u_y  Fixed          0.0000         121.7073
   9 theta   Free        0.007828           0.0000


In [95]:
# -------------------------
# Display Element Results
# -------------------------
# Compute and print the element-level results for each member.

print_element(1, u_global, m1, T1, k1, Qf1, disp_in_mm=True, dec=1, rad_dec=6)
print_element(2, u_global, m2, T2, k2, Qf2, disp_in_mm=True, dec=1, rad_dec=6)


E1
u [mm,rad]: [-0.3, 0.0, -0.004317, 0.0, 0.0, 0.000000]
v [mm,rad]: [0.2, 0.2, -0.004317, 0.0, 0.0, 0.000000]
q [kN,kN·m]: [40.9, -101.6, -417.5, -40.9, -23.4, 26.6]

E2
u [mm,rad]: [-0.3, 0.0, -0.004317, 0.0, 0.0, 0.007828]
v [mm,rad]: [-0.3, 0.0, -0.004317, 0.0, 0.0, 0.007828]
q [kN,kN·m]: [-56.8, 166.3, 267.5, 56.8, 121.7, 0.0]
